# Where Am I? Position in Attention, from Sinusoids to RoPE

The full code of [Where Am I? Position in Attention, from Sinusoids to RoPE](https://residual-stream.lvtan71.workers.dev/posts/rope-and-positional-encoding/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/rope-and-positional-encoding) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rope-and-positional-encoding.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.11.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "rope-and-positional-encoding"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `permutation_rope.py`

Checks that attention without positions is permutation-equivariant and that a causal mask breaks this; RoPE as a rotation of dimension pairs, whose query-key score depends only on the distance and which keeps a vector's length.

In [ ]:
!{sys.executable} permutation_rope.py

The post printed:

```text
attention(permuted input) == permuted attention(input): True
the same, with a causal mask: False
query at   3, key at   5 (distance  2): score -6.3341
query at  10, key at  12 (distance  2): score -6.3341
query at 100, key at 102 (distance  2): score -6.3341
query at   3, key at  10 (distance  7): score -5.9618
rotation keeps length: |q| = 4.4482, |rope(q, 57)| = 4.4482
```

## `all_runs.py`

Runs `train_position.py` for both tasks, seven schemes and seeds 0 and 1: 28 runs, one line each.

In [ ]:
!{sys.executable} all_runs.py

The post printed:

```text
lag  none         seed 0: length 32 / 64 / 128: 0.522 / 0.339 / 0.240; at 128, positions past 32: 0.158; scores x log128/log32: 0.239
lag  none         seed 1: length 32 / 64 / 128: 0.526 / 0.341 / 0.243; at 128, positions past 32: 0.163; scores x log128/log32: 0.243
lag  sinusoid     seed 0: length 32 / 64 / 128: 1.000 / 0.539 / 0.315; at 128, positions past 32: 0.113; scores x log128/log32: 0.317
lag  sinusoid     seed 1: length 32 / 64 / 128: 1.000 / 0.531 / 0.307; at 128, positions past 32: 0.095; scores x log128/log32: 0.301
lag  learned      seed 0: length 32 / 64 / 128: 1.000 / 0.520 / 0.292; at 128, positions past 32: 0.078; scores x log128/log32: 0.291
lag  learned      seed 1: length 32 / 64 / 128: 1.000 / 0.516 / 0.293; at 128, positions past 32: 0.079; scores x log128/log32: 0.291
lag  rope         seed 0: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  rope         seed 1: length 32 / 64 / 128: 1.000 / 1.000 / 0.999; at 128, positions past 32: 0.999; scores x log128/log32: 1.000
lag  rope:1000000 seed 0: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  rope:1000000 seed 1: length 32 / 64 / 128: 1.000 / 1.000 / 1.000; at 128, positions past 32: 1.000; scores x log128/log32: 1.000
lag  alibi:0.5    seed 0: length 32 / 64 / 128: 0.940 / 0.939 / 0.939; at 128, positions past 32: 0.938; scores x log128/log32: 0.878
lag  alibi:0.5    seed 1: length 32 / 64 / 128: 0.942 / 0.936 / 0.936; at 128, positions past 32: 0.938; scores x log128/log32: 0.867
lag  alibi:0.0625 seed 0: length 32 / 64 / 128: 0.698 / 0.634 / 0.605; at 128, positions past 32: 0.587; scores x log128/log32: 0.645
lag  alibi:0.0625 seed 1: length 32 / 64 / 128: 0.699 / 0.636 / 0.602; at 128, positions past 32: 0.582; scores x log128/log32: 0.643
seen none         seed 0: length 32 / 64 / 128: 0.997 / 0.959 / 0.808; at 128, positions past 32: 0.747; scores x log128/log32: 0.960
seen none         seed 1: length 32 / 64 / 128: 0.997 / 0.963 / 0.813; at 128, positions past 32: 0.754; scores x log128/log32: 0.960
seen sinusoid     seed 0: length 32 / 64 / 128: 1.000 / 0.901 / 0.725; at 128, positions past 32: 0.638; scores x log128/log32: 0.715
seen sinusoid     seed 1: length 32 / 64 / 128: 1.000 / 0.892 / 0.723; at 128, positions past 32: 0.633; scores x log128/log32: 0.715
seen learned      seed 0: length 32 / 64 / 128: 0.999 / 0.760 / 0.638; at 128, positions past 32: 0.524; scores x log128/log32: 0.641
seen learned      seed 1: length 32 / 64 / 128: 0.999 / 0.727 / 0.617; at 128, positions past 32: 0.498; scores x log128/log32: 0.627
seen rope         seed 0: length 32 / 64 / 128: 0.999 / 0.936 / 0.820; at 128, positions past 32: 0.762; scores x log128/log32: 0.844
seen rope         seed 1: length 32 / 64 / 128: 1.000 / 0.942 / 0.813; at 128, positions past 32: 0.753; scores x log128/log32: 0.845
seen rope:1000000 seed 0: length 32 / 64 / 128: 0.999 / 0.968 / 0.870; at 128, positions past 32: 0.827; scores x log128/log32: 0.892
seen rope:1000000 seed 1: length 32 / 64 / 128: 0.999 / 0.967 / 0.837; at 128, positions past 32: 0.785; scores x log128/log32: 0.877
seen alibi:0.5    seed 0: length 32 / 64 / 128: 0.933 / 0.877 / 0.752; at 128, positions past 32: 0.689; scores x log128/log32: 0.785
seen alibi:0.5    seed 1: length 32 / 64 / 128: 0.938 / 0.881 / 0.755; at 128, positions past 32: 0.698; scores x log128/log32: 0.791
seen alibi:0.0625 seed 0: length 32 / 64 / 128: 0.991 / 0.942 / 0.839; at 128, positions past 32: 0.790; scores x log128/log32: 0.652
seen alibi:0.0625 seed 1: length 32 / 64 / 128: 0.993 / 0.940 / 0.831; at 128, positions past 32: 0.780; scores x log128/log32: 0.663
```

## `wavelengths.py`

Wavelength of each RoPE dimension pair at head dimension 128, for bases 10,000 and 500,000, and how many pairs never complete a turn within 8,192 and 131,072 positions.

In [ ]:
!{sys.executable} wavelengths.py

The post printed:

```text
 pair        base 10,000      base 500,000
    0                6.3               6.3
    8               19.9              32.4
   16               62.8             167.1
   32              628.3           4,442.9
   48            6,283.2         118,142.8
   63           54,410.1       2,559,195.5
base 10,000: 14 of 64 pairs never finish one turn within 8,192 positions,  0 within 131,072
base 500,000: 29 of 64 pairs never finish one turn within 8,192 positions, 15 within 131,072
```